In [1]:
!nvidia-smi

Wed Mar 30 15:04:12 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.57.02    Driver Version: 512.15       CUDA Version: 11.6     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  On   | 00000000:4C:00.0  On |                  N/A |
|  0%   42C    P8    34W / 300W |   1227MiB / 11264MiB |     14%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
!nvcc --version

/bin/bash: nvcc: command not found


In [3]:
import os
import random
from datetime import datetime
import time
import itertools
import concurrent
from concurrent.futures import ThreadPoolExecutor
import cshogi.cli

In [4]:
os.chdir('..')
# os.chdir('..')

In [5]:
os.getcwd()

'/mnt/c/Users/hmats/workspace/DeepLearningShogi'

In [6]:
# pip install -e .

In [7]:
# os.listdir('//corgi/hmatsuya/workspace/Shogi/data/dl_data/hcpe')
# os.listdir('/mnt/corgi/hmatsuya/workspace/Shogi/data/dl_data/hcpe')

In [8]:
os.getcwd()

'/mnt/c/Users/hmats/workspace/DeepLearningShogi'

In [9]:
os.chdir('dlshogi')

In [10]:
#os.mkdir('../model')

In [11]:
#os.mkdir('../log')

## Train!

In [12]:
hcpelist = []
for root, dirs, files in os.walk('/home/hmatsuya/teacher'):
    for file in files:
        if 'hcpe' in file:
            hcpelist.append(os.path.join(root, file))
len(hcpelist)

7762

In [13]:
# resnet8_dropout
# # advantage: no value
# val_lambda: 0
# tanh
# polic_coef: 0.5

chunk_size = 38 * 20 * 1000 * 1000
activation='C:\\Anaconda3\\Scripts\\activate.bat'
test='/mnt/corgi/hmatsuya/workspace/Shogi/data/dl_data/hcpe/floodgate_teacher_uniq-test-01'

#! wsl ssh hmatsuya@corgi ls -lh /home/hmatsuya/workspace/Shogi/data/apery_teacher/

# resume = '-r ../model/state-2019'
resume = ''
# teacher_dir = '/mnt/corgi/hmatsuya/workspace/Shogi/data/apery_teacher/shuffled'
# filelist = os.listdir(teacher_dir)
# hcpelist = list(filter(lambda f: '.hcpe' in f, filelist))
files = []
project = "test-critic"
test_name = "train3g"
name = f'{test_name}'
run_id = f'{name}.{time.time()}'
run_id =  'train3g.1630241313.0797133'
i = 77
if i == 0:
    resume = ''
else:
    resume = f'-m ../model/model-{name}-{i-1} -r ../model/checkpoint-{name}-{i-1}.pth'

sum_size = 0
for file in random.sample(hcpelist, len(hcpelist)):
    
    files.append(file)
    sum_size += os.path.getsize(file)
    if sum_size < 1024 * 1024 * 1024 * 3:
        continue

    sum_size = 0
    
    teacher=' '.join([f'{f}' for f in files])
    
    model=f'../model/model-{name}-{i}'
    checkpoint=f'../model/checkpoint-{name}-{i}.pth'
    log=f'../log/{name}.txt'
    ! python -u train.py {teacher} {test} --model {model} --checkpoint {checkpoint} {resume} --use_result_critic --critic_lambda 0.5 --val_lambda 0.1 --lr 0.0001 --weight_decay 0.000001 --use_average --use_evalfix --log {log} --use_amp --project {project} --run_id {run_id}

    files.clear()
    resume = f'-r {checkpoint} -m {model}'
    i += 1


2021/09/28 23:46:02	INFO	network resnet10_swish
2021/09/28 23:46:02	INFO	batchsize=1024
2021/09/28 23:46:02	INFO	lr=0.0001
2021/09/28 23:46:02	INFO	weight_decay=1e-06
2021/09/28 23:46:02	INFO	val_lambda=0.1
wandb: Currently logged in as: hmatsuya (use `wandb login --relogin` to force relogin)
wandb: wandb version 0.12.2 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.11.2
wandb: Syncing run train3g
wandb:  View project at https://wandb.ai/hmatsuya/test-critic
wandb:  View run at https://wandb.ai/hmatsuya/test-critic/runs/train3g.1630241313.0797133
wandb: Run data is saved locally in /mnt/c/Users/hmats/workspace/DeepLearningShogi/dlshogi/wandb/run-20210928_234608-train3g.1630241313.0797133
wandb: Run `wandb offline` to turn off syncing.

2021/09/28 23:46:14	INFO	use amp
2021/09/28 23:46:14	INFO	use evalfix
2021/09/28 23:46:14	INFO	temperature=1.0
2021/09/28 23:46:14	INFO	Loading the model from ../model/model-train3g-7

In [14]:
os.getcwd()

'/mnt/c/Users/hmats/workspace/DeepLearningShogi/dlshogi'

In [1]:
test_name = "result_critic01"
name = f'{test_name}'
model=f'../model/model-{name}-15'

# Convert to ONNX
! python ../dlshogi/convert_model_to_onnx.py --network resnet10_swish ../model/{model} ../model/{model}.onnx
! scp ../model/{model}.onnx corgi:/home/hmatsuya/workspace/Shogi/dlcobra/model/

graph(%input1 : Float(1:5022, 62:81, 9:9, 9:1, requires_grad=0, device=cuda:0),
      %input2 : Float(1:4617, 57:81, 9:9, 9:1, requires_grad=0, device=cuda:0),
      %l1_1_1.weight : Float(192:558, 62:9, 3:3, 3:1, requires_grad=1, device=cuda:0),
      %l1_1_2.weight : Float(192:62, 62:1, 1:1, 1:1, requires_grad=1, device=cuda:0),
      %l1_2.weight : Float(192:57, 57:1, 1:1, 1:1, requires_grad=1, device=cuda:0),
      %l22.weight : Float(27:192, 192:1, 1:1, 1:1, requires_grad=1, device=cuda:0),
      %l22_2.bias : Float(2187:1, requires_grad=1, device=cuda:0),
      %l23_v.weight : Float(256:2187, 2187:1, requires_grad=1, device=cuda:0),
      %l23_v.bias : Float(256:1, requires_grad=1, device=cuda:0),
      %l24_v.weight : Float(1:256, 256:1, requires_grad=1, device=cuda:0),
      %l24_v.bias : Float(1:1, requires_grad=1, device=cuda:0),
      %norm1.weight : Float(192:1, requires_grad=1, device=cuda:0),
      %norm1.bias : Float(192:1, requires_grad=1, device=cuda:0),
      %norm1.r